# Train YOLOv8n on COCO person subset (Colab-ready)

This notebook downloads COCO 2017 val set, extracts person annotations, trains a `yolov8n` model for a quick demo, and exports an ONNX model you can use with the STM32 generator script.

Notes:
- This is intended for Colab / GPU. Adjust `epochs` and `batch` for your VM.
- For full training use the COCO train split; this notebook uses `val2017` to keep download size smaller for demo.

In [ ]:
# Install dependencies
!pip install -q ultralytics pycocotools wget

In [ ]:
# Download COCO val2017 (images) and annotations (smaller than full train)
import os
os.makedirs('coco', exist_ok=True)
%cd coco
!wget -q http://images.cocodataset.org/zips/val2017.zip -O val2017.zip
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O annotations_trainval2017.zip
!unzip -q val2017.zip
!unzip -q annotations_trainval2017.zip
%cd ..

In [ ]:
# Convert COCO -> YOLO (person only)
import json, os, shutil
from pathlib import Path
coco_root = 'coco'
out_dir = 'person_dataset'
os.makedirs(out_dir, exist_ok=True)
# load val annotations (we'll use val as both train/val for demo)
ann_val = os.path.join(coco_root, 'annotations', 'instances_val2017.json')
images_dir = os.path.join(coco_root, 'val2017')
with open(ann_val, 'r', encoding='utf8') as f:
    coco = json.load(f)
images = {img['id']: img for img in coco['images']}
anns_by_image = {}
for ann in coco['annotations']:
    if ann.get('category_id') != 1:
        continue
    anns_by_image.setdefault(ann['image_id'], []).append(ann)
# prepare output folders
train_images = os.path.join(out_dir, 'images', 'train')
val_images = os.path.join(out_dir, 'images', 'val')
train_labels = os.path.join(out_dir, 'labels', 'train')
val_labels = os.path.join(out_dir, 'labels', 'val')
os.makedirs(train_images, exist_ok=True)
os.makedirs(val_images, exist_ok=True)
os.makedirs(train_labels, exist_ok=True)
os.makedirs(val_labels, exist_ok=True)
# For demo, split 80/20 from val set containing persons
items = list(anns_by_image.items())
split_idx = int(len(items) * 0.8)
train_items = items[:split_idx]
val_items = items[split_idx:]
def write_items(items, out_images_dir, out_labels_dir):
    copied = 0
    for img_id, anns in items:
        img = images[img_id]
        fname = img['file_name']
        src = os.path.join(images_dir, fname)
        if not os.path.exists(src):
            continue
        dst = os.path.join(out_images_dir, fname)
        shutil.copy2(src, dst)
        w = img['width']
        h = img['height']
        lines = []
        for a in anns:
            x, y, bw, bh = a['bbox']
            x_c = x + bw/2.0
            y_c = y + bh/2.0
            lines.append(f'0 {x_c/w:.6f} {y_c/h:.6f} {bw/w:.6f} {bh/h:.6f}')
        label_path = os.path.join(out_labels_dir, Path(fname).with_suffix('.txt').name)
        with open(label_path, 'w', encoding='utf8') as lf:
            lf.write('
'.join(lines))
        copied += 1
    return copied
ntrain = write_items(train_items, train_images, train_labels)
nval = write_items(val_items, val_images, val_labels)
print(f'Created dataset: {ntrain} train images, {nval} val images in {out_dir}')

In [ ]:
# Write data_person.yaml used by Ultralytics
data_yaml = '''train: person_dataset/images/train
val: person_dataset/images/val
nc: 1
names: ['person']
'''
with open('data_person.yaml','w') as f:
    f.write(data_yaml)
print('Wrote data_person.yaml')

In [ ]:
# Train a tiny model (short run for demo). Increase epochs for real training.
# Use the Ultralytics CLI if available
!yolo detect train model=yolov8n.pt data=data_person.yaml epochs=15 imgsz=320 batch=16

In [ ]:
# Export best weights to ONNX (after training completes)
!yolo export model=runs/detect/train/weights/best.pt format=onnx imgsz=320 || true
# show exported file if exists
!ls -l runs/detect/train/weights || true
!ls -l *.onnx || true